In [ ]:
!pip install catboost

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Part 1 ---------------------------------------------------------


data_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(data_path)



In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(6,4))
df["Delivery_Time"].hist(bins=30)
plt.xlabel("Delivery_Time")
plt.ylabel("count")
plt.title("Delivery time distribution")
plt.show()

In [ ]:
# Task 1: Write your code here:# Part 2 ---------------------------------------------------------
df = df.drop(columns=["Order_ID"])

In [ ]:
# Task 2: Write your code here:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object", "category"]).columns
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])



In [ ]:
# Task 3: Write your code here:
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [ ]:
# Task 5: Write your code here:
target_col = "Delivery_Time"
feature_cols = [c for c in df.columns if c != target_col]

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

In [ ]:
# Task 6: Write your code here:
print("Target is continuous → imbalance concept not applicable.")


In [ ]:
# Task 1: Write your code here:# Part 3 ---------------------------------------------------------
X = df[feature_cols].values
y = df[target_col].values

In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)

print("MAE scores:", mae_scores)
print("Average MAE:", np.mean(mae_scores))

In [ ]:
# Task 1: Write your code here:# Part 4 ---------------------------------------------------------
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(8,5))
plt.bar(range(len(feature_cols)), importances[sorted_idx])
plt.xticks(range(len(feature_cols)),
           np.array(feature_cols)[sorted_idx],
           rotation=90)
plt.title("Feature importance (RandomForest)")
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred_full = model.predict(X)

plt.figure(figsize=(6,4))
plt.hist(y_pred_full, bins=30)
plt.xlabel("Predicted delivery_time")
plt.ylabel("count")
plt.title("Predicted delivery time distribution")
plt.show()


In [ ]:
# Task Bonus: Write your code here:# Part 5 (Bonus) -----------------------------------------------

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
    cd= CatBoostRegressor(
        depth=6,
        learning_rate=0.05,
        n_estimators=500,
        verbose=False,
        random_state=42
    )

    rf.fit(X_train, y_train)
    cd.fit(X_train, y_train)

    rf_pred = rf.predict(X_val)
    cd_pred = cd.predict(X_val)

    y_pred_ens = (rf_pred + cd_pred) / 2.0

    mae = mean_absolute_error(y_val, y_pred_ens)
    ensemble_mae_scores.append(mae)

print("Ensemble MAE scores:", ensemble_mae_scores)
print("Average Ensemble MAE:", np.mean(ensemble_mae_scores))
